# Movement-strategy validation

**Question.** On a fixed mechanism, how much does quantising the commanded
direction degrade reproduction of a commanded path?

**Scope.** This study is standalone and concerns the control strategy only.
Everything is in *controller units* (command counts). Actuator calibration -
servo ticks, spool radius, dead zone, latency - is out of scope and belongs to
`validetion/Servomotor` and `validetion/Servo+thimble`. Do not mix those
numbers in here.

**Design.** One kinematic model (`KinematicModel.IK`) held fixed; the only
variable is the strategy:

| strategy | direction quantisation |
|---|---|
| `CARDINAL` | 4-way |
| `CARDINAL_DIAGONAL` | 8-way |
| `FREE_FORM` | none (reference) |

`MovementStrategy.IK` is deliberately absent: it is `FREE_FORM` paired with the
IK model, i.e. a *model* choice, not a strategy. Including it would vary the
model and the quantisation at once.

**Limitation.** Commands are decoded with the model that generated them, so
this measures strategy-induced path error, not physical mechanism accuracy.
That FK genuinely inverts the controller's IK path is verified separately in
`tests/test_wire_forward_kinematics.py`.

All computation lives in `analysis.py` and `figures.py`; this notebook only
calls them, so there is exactly one implementation.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

from analysis import StudyConfig, run_study
from figures import plot_error_decomposition, plot_motor_commands, plot_reconstructed_circles

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

config = StudyConfig()
samples, metrics = run_study(config)

print(f"{len(samples) // len(config.strategies)} samples per strategy, "
      f"radius {config.radius:.0f} controller units, "
      f"model fixed at '{config.kinematic_model.value}'.")

301 samples per strategy, radius 160 controller units, model fixed at 'ik'.


## Error decomposition

    ideal target --(strategy quantisation)--> quantised target
                 --(model + integer truncation)--> reconstructed point

`quantisation` is what the strategy costs. `execution` is the shared noise
floor and should be near-identical across strategies - if it is not, something
other than the strategy is varying.

In [2]:
metrics[["quantisation_rms", "quantisation_max",
         "execution_rms", "execution_max",
         "total_rms", "total_max"]].round(2)

,quantisation_rms,quantisation_max,execution_rms,execution_max,total_rms,total_max
strategy,,,,,,
cardinal,61.07,120.34,2.75,7.04,60.87,121.85
cardinal_diagonal,32.64,72.48,2.96,7.04,32.61,74.04
free_form,0.00,0.00,3.52,10.43,3.52,10.43


## Shape metrics, and why they mislead

Quantisation is radius-preserving, so every reconstructed point sits on the
commanded radius. Aspect ratio and circle-fit RMS therefore score all three
strategies as near-perfect circles while the point-to-point error differs by
more than an order of magnitude. This is the reason the study reports
point-to-point error as its primary metric.

In [3]:
metrics[["radial_rms", "circle_fit_rms", "aspect_ratio",
         "closure_error", "fk_invalid_steps"]].round(3)

,radial_rms,circle_fit_rms,aspect_ratio,closure_error,fk_invalid_steps
strategy,,,,,
cardinal,1.912,0.323,1.004,2.853,0
cardinal_diagonal,2.167,0.768,1.004,2.853,0
free_form,3.143,1.693,0.997,0.000,0


## Command effort

Comparable now that one kinematic model is used for every row. Amplitude is
near-identical across strategies; the genuine difference is `max_step_jump` -
the discontinuity when a quantised direction snaps to a new sector.

In [4]:
metrics[["peak_abs_command", "rms_command",
         "total_motor_travel", "max_step_jump"]].round(3)

,peak_abs_command,rms_command,total_motor_travel,max_step_jump
strategy,,,,
cardinal,45.0,26.484,577.0,72.0
cardinal_diagonal,50.0,26.397,603.0,45.0
free_form,53.0,26.365,622.0,2.0


## Figures

In [5]:
for path in (plot_reconstructed_circles(samples, config),
             plot_error_decomposition(samples, metrics, config),
             plot_motor_commands(samples, config)):
    print("saved", path.name)
    display(plt.imread(path).shape)

saved reconstructed_circles_by_strategy.png


(885, 2411, 4)

saved error_decomposition.png


(731, 2009, 4)

saved motor_commands_by_strategy.png


(1372, 1784, 4)